In [5]:
import pickle
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from pathlib import Path


# Works when Jupyter is started from the project root or from notebooks/.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

# Load Home Credit
hc = pd.read_csv(PROJECT_ROOT / 'data/home_credit/application_train.csv')
print("Original shape:", hc.shape)

# Drop columns with >50% missing values
missing_pct = hc.isnull().sum() / len(hc)
cols_to_drop = missing_pct[missing_pct > 0.5].index
hc = hc.drop(columns=cols_to_drop)
print("After dropping high-missing columns:", hc.shape)
print("Dropped", len(cols_to_drop), "columns")


Original shape: (307511, 122)
After dropping high-missing columns: (307511, 81)
Dropped 41 columns


In [6]:
# Fix the DAYS_EMPLOYED anomaly (365243 = unemployed)
hc['DAYS_EMPLOYED'] = hc['DAYS_EMPLOYED'].replace(365243, np.nan)
print("Fixed DAYS_EMPLOYED anomaly")

# Encode categorical columns
from sklearn.preprocessing import LabelEncoder

cat_cols = hc.select_dtypes(include=['object']).columns.tolist()
print(f"Found {len(cat_cols)} categorical columns")

le = LabelEncoder()
for col in cat_cols:
    hc[col] = hc[col].astype(str)
    hc[col] = le.fit_transform(hc[col])

print("Encoded all categorical columns")

Fixed DAYS_EMPLOYED anomaly
Found 13 categorical columns
Encoded all categorical columns


In [7]:
# Fill remaining missing values with median
hc = hc.fillna(hc.median(numeric_only=True))
print("Remaining missing values:", hc.isnull().sum().sum())

# Features and target
X_hc = hc.drop(['TARGET', 'SK_ID_CURR'], axis=1)
y_hc = hc['TARGET']

print("Feature shape:", X_hc.shape)

# Split
X_hc_train, X_hc_test, y_hc_train, y_hc_test = train_test_split(
    X_hc, y_hc, test_size=0.2, random_state=42, stratify=y_hc
)

# Scale
scaler_hc = StandardScaler()
X_hc_train = scaler_hc.fit_transform(X_hc_train)
X_hc_test = scaler_hc.transform(X_hc_test)

# Save
data_path = PROJECT_ROOT / "data/clean/hc_data.pkl"
data_path.parent.mkdir(parents=True, exist_ok=True)
with open(data_path, "wb") as f:
    pickle.dump((X_hc_train, X_hc_test, y_hc_train, y_hc_test, X_hc.columns.tolist()), f)

print("Home Credit saved!")
print("Train:", X_hc_train.shape, "| Test:", X_hc_test.shape)


Remaining missing values: 0
Feature shape: (307511, 79)
Home Credit saved!
Train: (246008, 79) | Test: (61503, 79)


In [9]:
import pickle
import os

models_dir = PROJECT_ROOT / "models"
models_dir.mkdir(parents=True, exist_ok=True)

with open(models_dir / "hc_scaler.pkl", "wb") as f:
    pickle.dump(scaler_hc, f)

print(" Scaler saved successfully!")


 Scaler saved successfully!


In [ ]:
# The saved inference artifacts include the exact feature order.
with open(models_dir / "hc_feature_names.pkl", "wb") as f:
    pickle.dump(X_hc.columns.tolist(), f)

print(f"Saved {len(X_hc.columns)} feature names to models/hc_feature_names.pkl")
